In [ ]:
%py
# Databricks PySpark script for d_product_revenue_clone table with masked invoice_number

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, length

# Initialize Spark session
spark = SparkSession.builder.appName("Invoice Masking").getOrCreate()

# Load d_product_revenue table with error handling
try:
    # Load the data from the specified Unity Catalog table
    d_product_revenue_df = spark.read.table("purgo_databricks.qa_1.d_product_revenue")
except Exception as e:
    print(f"Error loading d_product_revenue table: {e}")
    raise

# Mask the last 4 digits of the invoice_number in the dataframe
masked_df = d_product_revenue_df.withColumn(
    "invoice_number",
    # Apply masking logic for valid invoice numbers
    when(
        col("invoice_number").isNotNull() & (col("invoice_number").rlike("^[0-9A-Za-z]+$")) & (length(col("invoice_number")) >= 4),
        # Preserve the first part of the invoice_number and mask the last 4 digits
        col("invoice_number").substr(1, length(col("invoice_number")) - 4) + "****"
    # Keep the original invoice_number if conditions are not met
    ).otherwise(col("invoice_number"))
)

# Drop the clone table if it exists
spark.sql("DROP TABLE IF EXISTS purgo_databricks.qa_1.d_product_revenue_clone")

# Create a clone table and save the masked data in Delta format
masked_df.write.format("delta").saveAsTable("purgo_databricks.qa_1.d_product_revenue_clone")

# Terminate Spark session
spark.stop()